# 03 · 城市天際線-動手做(只調高度)
**鎖死 footprint 與角色標籤,只放開高度這一個自由度**,看權力怎麼改寫天際線。
和新加坡 OSM 版同一手感——但跑**上海真實數據、實測高度**(陸家嘴最高 338m)。不用寫代碼,逐格執行即可。

> 你可以改三處(都用記事本就能開):
> - `config.py` 的 `SLUG` —— 換站點(lujiazui / caoyang / yuyuan)。改完重跑下面「準備」格。
> - `shanghai_lookup.yaml` —— 誰算誰的。改完重跑 **Step 2**。
> - `power_scenarios.yaml` —— 每個角色的高度政策(mult / cap_m / _mode)。改完重跑 **Step 3、4**。

## 怎麼執行
- 點一格,按 **Shift+Enter** 執行;或選單 Run → Run All Cells 從頭跑。
- 結果與圖會顯示在該格下面。代碼都在 `engine/` 文件夾,平時不用打開。

In [ ]:
# 讓 notebook 找到 engine 裏的代碼(這格不用改,直接執行)
import sys, os
_R = os.path.abspath("." if os.path.exists("engine") else "..")   # 包根(notebook 在 教學用notebook/ 子夹也找得到)
for _p in (_R, os.path.join(_R, "engine")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
# 清掉舊的引擎模塊緩存:改了 config/engine 後,重跑本格即刻生效(免重啟內核)
for _m in [k for k in list(sys.modules) if k in ("config", "common", "operators", "measure")
           or k == "plots" or k.startswith("plots.") or k == "prints" or k.startswith("prints.")]:
    del sys.modules[_m]
import config, common as C, plots, prints
prints.ready()
plots.capture("03")   # 開啟自動存圖:本冊每張圖都會存到 out/<slug>/Step_03/<時間戳>/

In [ ]:
df = C.current_buildings(config.SLUG)
df = C.assign_all(df)           # 貼上 stakeholder(改 shanghai_lookup.yaml 可變)
prints.headline(df)

## Step 0(可選):先看真實的地方
衛星底圖 + figure-ground(把我們的離散讀法疊在真實衛星上)。需要 `contextily` 和網絡,沒有就跳過。

In [ ]:
try:
    plots.satellite_figureground(df)     # 需聯網;失敗自動跳過
except Exception as e:
    prints.skipped("衛星", e)

## Step 1:讀建築(實測高度)
這步**還沒上色**,全灰,先看清原料:這批樓長什麼形狀、有多高。上海版高度**全是 AI 實測**,
不是樓層估算——所以纔有真正的超高層 token(右圖 >100m 的紅色)。

In [ ]:
plots.data_overview(df)

## Step 2:貼角色標籤
按 `shanghai_lookup.yaml` 的級聯查表,給每棟一個角色(詳見「02 映射」那本)。只加標籤,不動幾何與高度。

In [ ]:
prints.stakeholders(df)
plots.power_map(df)

## Step 3:權力情景(高度政策面板)
每個情景給各角色一個高度權重 `mult`、可選上限 `cap_m`,與模式 `_mode`:
- **conserve**(默認):總建築量守恆,只在角色之間**重分配**。
- **grow**:以現有高度爲地板,**只增不減**(都更新增),總量上升。

In [ ]:
scen = C.load_scenarios()
prints.scenarios(scen)
plots.policy_heatmap(scen)      # 紅=拔高 / 藍=壓低 / 白=照舊

## Step 4:套情景,只調高度 —— PAYOFF
同一批樓、同樣 footprint 與標籤,分別套幾個情景**只改高度**。因爲 footprint 與標籤鎖定,
天際線差異**完全來自權力配置**。conserve 總量守恆、只此消彼長;想看大變化用 grow 情景(如 `developer_renewal`)。

In [ ]:
SCEN_NAMES = ["current", "developer_led", "community_led", "state_eco"]   # 可改要比的情景
heights = {n: (C.scenario_heights(df, scen[n]) if n != "current" else df["height_m"]) for n in SCEN_NAMES}
plots.skyline_panels(df, heights)    # 4 連幅,同色階可橫比
plots.metrics(df, heights)           # 總GFA / 平均高 / 最高 / 各角色平均高

## Step 5:出 3D 量體 + OBJ(給 Rhino/GH)
把 footprint 按高度擠成 3D。屏幕預覽取佔地最大的若干棟(畫得快);OBJ 用全部樓、寫到 `out/<slug>/`。

In [ ]:
plots.city_3d(df, top=120)                          # 現狀 3D(取佔地最大的 120 棟,畫得快)
dev = df.copy(); dev["height_m"] = heights["developer_led"].values
plots.city_3d(dev, top=120)                         # 多一張:developer_led 情景的 3D 天際線
path, nv, nf = C.export_obj(df, config.SLUG)         # 全部樓擠成 OBJ → out/<slug>/
prints.obj_written(path, nv, nf)

## 動手練習
改一處、重跑對應步驟,看天際線變化:
- 換站點 → `config.py` 的 `SLUG`(陸家嘴↔曹楊↔豫園)→ 重跑「準備」格起。
- 改誰算誰的 → `shanghai_lookup.yaml` → 重跑 Step 2。
- 改高度政策 / 加情景 → `power_scenarios.yaml` → 重跑 Step 3、4。

> 想讓權力不止改高度、而是改**形態語法**(拆板成塔、細粒自建、向權力重心收攏)?
> 進下一本:[`04 進階-權力算子`]。